In [5]:
pip install --upgrade yfinance curl_cffi

Defaulting to user installation because normal site-packages is not writeable
  Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl.metadata (18 kB)
  Using cached markdown_it_py-4.0.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
Using cached curl_cffi-0.15.0-cp310-abi3-win_amd64.whl (1.7 MB)
Using cached markdown_it_py-4.0.0-py3-none-any.whl (87 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)

   -------- ------------------------------- 1/5 [markdown-it-py]
   ---------------- ----------------------- 2/5 [rich]
   ---------------- ----------------------- 2/5 [rich]
   ---------------- ----------------------- 2/5 [rich]
  Attempting uninstall: curl_cffi
   ---------------- ----------------------- 2/5 [rich]
    Found existing installation: curl_cffi 0.13.0
   ---------------- ----------------------- 2/5 [rich]
    Uninstalling curl_cffi-0.13.0:
   ---------------- ----------------------- 2/5 [rich]
      Successfully uninst

  You can safely remove it manually.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import yfinance as yf
import pandas as pd

# No pasar session — dejar que yfinance maneje curl_cffi solo
ticker = yf.Ticker("BIRD")

# Datos intraday hoy
hist_1d = ticker.history(period="1d", interval="2m")

if hist_1d.empty:
    print("⚠️ Sin datos — mercado cerrado o ticker no disponible")
else:
    print(f"✅ {len(hist_1d)} velas cargadas\n")
    print(hist_1d[["Close", "Volume"]].tail(10))

    # RSI con EWM (más estable que rolling para series cortas)
    close = hist_1d["Close"]
    window = 14 if len(close) >= 15 else max(len(close) - 1, 2)

    delta = close.diff()
    gain = delta.clip(lower=0).ewm(span=window, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(span=window, adjust=False).mean()
    rsi = 100 - (100 / (1 + gain / loss))

    nivel = rsi.iloc[-1]
    print(f"\nRSI({window}): {nivel:.1f}")
    if nivel > 80:   print("🔴 Sobrecompra extrema")
    elif nivel > 70: print("🟠 Sobrecompra")
    elif nivel < 30: print("🟢 Sobreventa")
    else:            print("🟡 Zona neutral")

✅ 33 velas cargadas

                             Close   Volume
Datetime                                   
2026-04-15 10:18:00-04:00  11.4000   917763
2026-04-15 10:20:00-04:00  10.8900  1343821
2026-04-15 10:22:00-04:00  10.9800   429466
2026-04-15 10:24:00-04:00  10.6550  1035966
2026-04-15 10:26:00-04:00  10.7699   505085
2026-04-15 10:28:00-04:00  10.8850  1089796
2026-04-15 10:30:00-04:00  11.6530  1921312
2026-04-15 10:32:00-04:00  13.2000  4183570
2026-04-15 10:34:00-04:00  13.5300  2427274
2026-04-15 10:36:00-04:00  11.5401   902936

RSI(14): 49.8
🟡 Zona neutral


In [11]:
import yfinance as yf
import pandas as pd
import numpy as np

ticker = yf.Ticker("BIRD")
df = ticker.history(period="1d", interval="2m")

# VWAP (Volume Weighted Average Price)
df["TP"] = (df["High"] + df["Low"] + df["Close"]) / 3
df["VWAP"] = (df["TP"] * df["Volume"]).cumsum() / df["Volume"].cumsum()

# Bollinger Bands (20 períodos, 2 desviaciones)
window_bb = min(20, len(df) - 1)
df["BB_mid"] = df["Close"].rolling(window_bb).mean()
df["BB_std"] = df["Close"].rolling(window_bb).std()
df["BB_upper"] = df["BB_mid"] + 2 * df["BB_std"]
df["BB_lower"] = df["BB_mid"] - 2 * df["BB_std"]

# RSI EWM
delta = df["Close"].diff()
gain = delta.clip(lower=0).ewm(span=14, adjust=False).mean()
loss = (-delta.clip(upper=0)).ewm(span=14, adjust=False).mean()
df["RSI"] = 100 - (100 / (1 + gain / loss))

# Señal respecto a VWAP
precio_actual = df["Close"].iloc[-1]
vwap_actual = df["VWAP"].iloc[-1]
bb_upper = df["BB_upper"].iloc[-1]
bb_lower = df["BB_lower"].iloc[-1]

print(f"Precio actual : ${precio_actual:.2f}")
print(f"VWAP          : ${vwap_actual:.2f}  → precio {'SOBRE' if precio_actual > vwap_actual else 'BAJO'} VWAP")
print(f"BB Superior   : ${bb_upper:.2f}")
print(f"BB Inferior   : ${bb_lower:.2f}")
print(f"RSI(14)       : {df['RSI'].iloc[-1]:.1f}")
print(df[["Close", "VWAP", "BB_upper", "BB_lower", "RSI"]].tail(5).round(2))

Precio actual : $11.98
VWAP          : $8.86  → precio SOBRE VWAP
BB Superior   : $12.90
BB Inferior   : $9.68
RSI(14)       : 54.4
                           Close  VWAP  BB_upper  BB_lower    RSI
Datetime                                                         
2026-04-15 10:30:00-04:00  11.65  8.57     11.91     10.29  66.17
2026-04-15 10:32:00-04:00  13.20  8.71     12.38      9.96  80.34
2026-04-15 10:34:00-04:00  13.53  8.80     12.89      9.68  82.18
2026-04-15 10:36:00-04:00  11.63  8.86     12.84      9.69  50.70
2026-04-15 10:38:00-04:00  11.98  8.86     12.90      9.68  54.36


In [ ]:
import time

def monitorear_bird(intervalo_seg=30, alertas={"caida": 10.50, "subida": 13.53}):
    """Corre en loop cada 2 minutos y alerta si se cruzan niveles clave."""
    print("🔍 Monitoreando BIRD... (Ctrl+C para detener)\n")
    
    while True:
        try:
            df = yf.Ticker("BIRD").history(period="1d", interval="2m")
            if df.empty:
                print("Sin datos"); time.sleep(intervalo_seg); continue

            precio   = df["Close"].iloc[-1]
            vwap     = ((df["High"]+df["Low"]+df["Close"])/3 * df["Volume"]).cumsum().iloc[-1] / df["Volume"].cumsum().iloc[-1]
            delta    = df["Close"].diff()
            gain     = delta.clip(lower=0).ewm(span=14, adjust=False).mean()
            loss     = (-delta.clip(upper=0)).ewm(span=14, adjust=False).mean()
            rsi      = (100 - (100 / (1 + gain / loss))).iloc[-1]
            hora     = df.index[-1].strftime("%H:%M")

            status = "✅" if precio > vwap else "⚠️"
            print(f"[{hora}] {status} Precio: ${precio:.2f} | VWAP: ${vwap:.2f} | RSI: {rsi:.1f}")

            # Alertas de nivel
            if precio <= alertas["caida"]:
                print(f"  🔴 ALERTA: Precio rompió soporte ${alertas['caida']}")
            if precio >= alertas["subida"]:
                print(f"  🟢 ALERTA: Precio rompió resistencia ${alertas['subida']}")

        except Exception as e:
            print(f"Error: {e}")
        
        time.sleep(intervalo_seg)

monitorear_bird()

🔍 Monitoreando BIRD... (Ctrl+C para detener)

[10:50] ✅ Precio: $13.40 | VWAP: $9.09 | RSI: 68.6
[10:52] ✅ Precio: $12.83 | VWAP: $9.12 | RSI: 57.9
[10:54] ✅ Precio: $13.26 | VWAP: $9.15 | RSI: 64.4
[10:56] ✅ Precio: $13.11 | VWAP: $9.17 | RSI: 60.0
[10:58] ✅ Precio: $13.02 | VWAP: $9.19 | RSI: 57.8
[11:00] ✅ Precio: $12.92 | VWAP: $9.21 | RSI: 55.6
[11:02] ✅ Precio: $12.99 | VWAP: $9.23 | RSI: 57.0
[11:04] ✅ Precio: $12.65 | VWAP: $9.24 | RSI: 48.9
[11:06] ✅ Precio: $13.36 | VWAP: $9.27 | RSI: 63.3
[11:08] ✅ Precio: $13.53 | VWAP: $9.29 | RSI: 66.5
[11:10] ✅ Precio: $14.13 | VWAP: $9.38 | RSI: 72.0
  🟢 ALERTA: Precio rompió resistencia $13.53
[11:12] ✅ Precio: $14.77 | VWAP: $9.41 | RSI: 79.6
  🟢 ALERTA: Precio rompió resistencia $13.53
[11:14] ✅ Precio: $16.20 | VWAP: $9.50 | RSI: 86.8
  🟢 ALERTA: Precio rompió resistencia $13.53
[11:16] ✅ Precio: $16.77 | VWAP: $9.64 | RSI: 88.8
  🟢 ALERTA: Precio rompió resistencia $13.53
[11:18] ✅ Precio: $18.42 | VWAP: $9.75 | RSI: 92.4
  🟢 ALERT

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import time
from datetime import datetime

# ── CONFIGURACIÓN ──────────────────────────────────────────────
TICKER        = "BIRD"
INTERVALO_SEG = 120       # cada 2 minutos
CAPITAL       = 500       # USD disponible para operar
RIESGO_PCT    = 0.02      # arriesgar máximo 2% del capital
STOP_LOSS_PCT = 0.07      # stop loss 7% bajo entrada

# Niveles clave (actualizar según la sesión)
RESISTENCIA   = 13.53
SOPORTE       = 9.68

# ── INDICADORES ────────────────────────────────────────────────
def calcular_indicadores(df):
    tp = (df["High"] + df["Low"] + df["Close"]) / 3
    df["VWAP"] = (tp * df["Volume"]).cumsum() / df["Volume"].cumsum()

    delta = df["Close"].diff()
    gain  = delta.clip(lower=0).ewm(span=14, adjust=False).mean()
    loss  = (-delta.clip(upper=0)).ewm(span=14, adjust=False).mean()
    df["RSI"] = 100 - (100 / (1 + gain / loss))

    w = min(20, len(df) - 1)
    df["BB_mid"]   = df["Close"].rolling(w).mean()
    df["BB_std"]   = df["Close"].rolling(w).std()
    df["BB_upper"] = df["BB_mid"] + 2 * df["BB_std"]
    df["BB_lower"] = df["BB_mid"] - 2 * df["BB_std"]

    df["Vol_ratio"] = df["Volume"] / df["Volume"].mean()
    return df

# ── SEÑALES ────────────────────────────────────────────────────
def evaluar_señal(row, prev_row, vol_promedio):
    precio    = row["Close"]
    rsi       = row["RSI"]
    vwap      = row["VWAP"]
    vol_ratio = row["Vol_ratio"]
    prev_precio = prev_row["Close"]

    señal = "ESPERAR"
    razon = []

    ruptura_resistencia = prev_precio < RESISTENCIA and precio >= RESISTENCIA
    volumen_confirmado  = vol_ratio >= 1.8
    rsi_ok_entrada      = 55 <= rsi <= 75
    sobre_vwap          = precio > vwap

    if ruptura_resistencia and volumen_confirmado and rsi_ok_entrada and sobre_vwap:
        señal = "🚀 ENTRADA"
        razon = [
            f"Ruptura de resistencia ${RESISTENCIA}",
            f"Volumen {vol_ratio:.1f}x el promedio (confirmado)",
            f"RSI {rsi:.1f} — momentum sin sobrecompra",
        ]
    elif rsi > 88:
        señal = "🔴 SOBRECOMPRA EXTREMA"
        razon = [f"RSI {rsi:.1f} — zona de distribución institucional"]
    elif precio < SOPORTE:
        señal = "🔴 ROMPE SOPORTE"
        razon = [f"Precio bajo ${SOPORTE} — stop loss activado"]
    elif ruptura_resistencia and not volumen_confirmado:
        señal = "⚠️  RUPTURA SIN VOLUMEN"
        razon = [f"Volumen solo {vol_ratio:.1f}x — esperar confirmación"]

    stop_loss        = precio * (1 - STOP_LOSS_PCT)
    riesgo_usd       = CAPITAL * RIESGO_PCT
    riesgo_x_accion  = precio - stop_loss
    acciones         = int(riesgo_usd / riesgo_x_accion) if riesgo_x_accion > 0 else 0
    capital_req      = acciones * precio

    return señal, razon, stop_loss, acciones, capital_req

# ── MONITOR PRINCIPAL ──────────────────────────────────────────
def monitorear(ticker_sym=TICKER):
    print(f"🔍 Monitor iniciado para {ticker_sym}")
    print(f"   Capital: ${CAPITAL} | Riesgo máx: {RIESGO_PCT*100:.0f}% | Stop: {STOP_LOSS_PCT*100:.0f}%")
    print(f"   Resistencia: ${RESISTENCIA} | Soporte: ${SOPORTE}")
    print("─" * 65)

    while True:
        try:
            df = yf.Ticker(ticker_sym).history(period="1d", interval="2m")
            if df.empty or len(df) < 3:
                print("⚠️  Sin datos"); time.sleep(INTERVALO_SEG); continue

            df       = calcular_indicadores(df)
            row      = df.iloc[-1]
            prev_row = df.iloc[-2]
            hora     = df.index[-1].strftime("%H:%M")
            precio   = row["Close"]

            señal, razon, stop_loss, acciones, capital_req = evaluar_señal(
                row, prev_row, df["Volume"].mean()
            )

            print(f"[{hora}] ${precio:.2f}  VWAP:${row['VWAP']:.2f}  RSI:{row['RSI']:.1f}  Vol:{row['Vol_ratio']:.1f}x  →  {señal}")

            if razon:
                for r in razon:
                    print(f"         • {r}")
                if señal == "🚀 ENTRADA":
                    print(f"         ──────────────────────────────")
                    print(f"         Stop loss  : ${stop_loss:.2f}")
                    print(f"         Acciones   : {acciones}  (${capital_req:.0f})")
                    print(f"         Riesgo máx : ${CAPITAL*RIESGO_PCT:.0f} USD")
                    print(f"         ⏱  ACTÚA EN LOS PRÓXIMOS 2 MIN")
                    print(f"         ──────────────────────────────")

        except Exception as e:
            print(f"[ERROR] {e}")

        time.sleep(INTERVALO_SEG)

if __name__ == "__main__":
    monitorear()

🔍 Monitor iniciado para BIRD
   Capital: $500 | Riesgo máx: 2% | Stop: 7%
   Resistencia: $13.53 | Soporte: $9.68
─────────────────────────────────────────────────────────────────
[11:42] $19.21  VWAP:$11.41  RSI:51.2  Vol:0.2x  →  ESPERAR
[11:44] $18.70  VWAP:$11.45  RSI:47.7  Vol:0.2x  →  ESPERAR
[11:46] $19.74  VWAP:$11.53  RSI:55.3  Vol:0.4x  →  ESPERAR
[11:48] $19.33  VWAP:$11.58  RSI:51.2  Vol:0.2x  →  ESPERAR
[11:50] $20.19  VWAP:$11.61  RSI:58.1  Vol:0.1x  →  ESPERAR
[11:52] $20.18  VWAP:$11.66  RSI:58.3  Vol:0.2x  →  ESPERAR
[11:54] $19.76  VWAP:$11.73  RSI:51.8  Vol:0.2x  →  ESPERAR
